
# Lake Michigan SST Time Series and Spatial Analysis Examples



## Import packages (if you don't have any of these, just install them as what you learned before)

These packages handle:
- NetCDF reading (`netCDF4`)
- numerical work (`numpy`)
- plotting (`matplotlib`)
- interpolation (`scipy.interpolate.griddata`)
- map features (`cartopy`)
- geometric masking (`shapely`)


In [ ]:

# Read NetCDF data files
import netCDF4 as nc

# Basic numerical tools
import numpy as np

# Plotting
from matplotlib import pyplot as plt

# Map plotting tools
import cartopy
import cartopy.crs as ccrs

# Interpolation for regridding
from scipy.interpolate import griddata

# Geometry tools for point-in-polygon tests
from shapely.geometry import Point

# File and path utilities
import os



## Define the input data path

Update this path to where the data is saved as what you did before


In [ ]:

# Path to the directory containing monthly SST netCDF files
path = '/proj/argon/yguo/Data/GL_ACSPO_AVHRR_Daily_Merge_SST/merged_monthly/'



## Build a monthly Lake Michigan mean SST time series

This loop reads all monthly files, masks Lake Michigan, and computes a lake-average SST.


In [ ]:

# Create an empty list to store the monthly mean SST values
sst_michigan_mean = []

# Create an empty list to store the corresponding dates
dates = []

# Loop through the available years
for yr in range(2021, 2026):

    # Loop through all months
    for mon in range(1, 13):

        # Skip months before June 2021
        if yr == 2021 and mon < 6:
            continue

        # Skip months after August 2025
        if yr == 2025 and mon > 8:
            continue

        # Build the filename
        filename = f'GL_ACSPO_AVHRR_merged_monthly_SST_{yr}{mon:02d}.nc'
        filepath = os.path.join(path, filename)

        # Skip if the file does not exist
        if not os.path.exists(filepath):
            print('Missing file:', filepath)
            continue

        # Open the monthly file
        dataset = nc.Dataset(filepath, 'r')

        # Read longitude, latitude, and SST
        lon = dataset.variables['longitude'][:]
        lat = dataset.variables['latitude'][:]
        sst = dataset.variables['sea_surface_temperature'][:] + 27315.0

        # Close the file
        dataset.close()

        # Create a Lake Michigan mask on the native satellite grid
        lake_mask = make_lake_mask(lon, lat, scale="50m")

        # Keep only Lake Michigan values
        sst_lake = np.where(lake_mask, sst, np.nan)

        # Compute the mean over Lake Michigan and store it
        sst_michigan_mean.append(np.nanmean(sst_lake))

        # Store the date string
        dates.append(f'{yr}-{mon:02d}')

# Convert the list into a numpy array
sst_michigan_mean = np.array(sst_michigan_mean)

# Print the number of time steps
print('Number of time steps =', len(sst_michigan_mean))


In [ ]:

# Plot the raw Lake Michigan monthly SST time series
plt.figure(figsize=(10, 4))
plt.plot(dates, sst_michigan_mean, 'k-', linewidth=2)
plt.xticks(rotation=45)
plt.ylabel('Lake-mean SST')
plt.title('Lake Michigan monthly SST time series')
plt.tight_layout()
plt.show()



## Remove the seasonal cycle using harmonic analysis

We fit a simple annual harmonic model with:
- an intercept
- cosine term
- sine term

Then we subtract the fitted seasonal cycle from the original series to obtain anomalies.


In [ ]:

# Create a time index 0, 1, 2, ..., N-1
t = np.arange(len(sst_michigan_mean))

# Build the design matrix for harmonic regression
# Column 1: intercept
# Column 2: annual cosine term
# Column 3: annual sine term
X = np.column_stack([
    np.ones(len(t)),
    np.cos(2 * np.pi * t / 12),
    np.sin(2 * np.pi * t / 12)
])

# Solve the least-squares regression problem
coeffs, _, _, _ = np.linalg.lstsq(X, sst_michigan_mean, rcond=None)

# Reconstruct the fitted seasonal cycle
seasonal = X @ coeffs

# Subtract the seasonal cycle from the original series
sst_anom_harmonic = sst_michigan_mean - seasonal

# Print the fitted coefficients
print('Regression coefficients =', coeffs)


In [ ]:

# Plot the original series and the fitted seasonal cycle
plt.figure(figsize=(10, 4))
plt.plot(dates, sst_michigan_mean, 'k-', linewidth=2, label='Original SST')
plt.plot(dates, seasonal, 'r-', linewidth=2, label='Fitted seasonal cycle')
plt.xticks(rotation=45)
plt.ylabel('SST')
plt.title('Harmonic seasonal fit')
plt.legend()
plt.tight_layout()
plt.show()

# Plot the anomaly series
plt.figure(figsize=(10, 4))
plt.plot(dates, sst_anom_harmonic, 'b-', linewidth=2)
plt.axhline(0, color='k', linewidth=0.8)
plt.xticks(rotation=45)
plt.ylabel('SST anomaly')
plt.title('Lake Michigan SST anomaly after harmonic deseasonalization')
plt.tight_layout()
plt.show()



## Read climate index time series remotely and compare with Lake Michigan SST anomalies

This cell downloads monthly:
- Niño 3.4 anomaly (ENSO)
- PDO
- NAO

Then it aligns those monthly series with the Lake Michigan monthly dates.


In [ ]:

# Remote file reading
from urllib.request import urlopen

# Use the SST anomaly series for climate-mode comparison
sst_use = np.array(sst_anom_harmonic, dtype=float)
dates_array = np.array(dates)

def read_psl_table_from_url(url):
    # Download text content from the URL
    text = urlopen(url).read().decode("utf-8", errors="ignore")

    # Split text into lines
    lines = text.splitlines()

    # Create empty output lists
    out_year = []
    out_month = []
    out_val = []

    # Read each line
    for line in lines:
        parts = line.strip().split()

        # Expect lines with: year + 12 monthly values
        if len(parts) < 13:
            continue

        # Skip non-data lines
        try:
            year = int(parts[0])
        except ValueError:
            continue

        # Read the 12 monthly values
        for m in range(12):
            try:
                val = float(parts[m + 1])
            except ValueError:
                val = np.nan

            # Replace common missing-value flags with NaN
            if val <= -99:
                val = np.nan

            out_year.append(year)
            out_month.append(m + 1)
            out_val.append(val)

    # Convert lists to numpy arrays
    return (
        np.array(out_year, dtype=int),
        np.array(out_month, dtype=int),
        np.array(out_val, dtype=float)
    )

def read_cpc_index_from_url(url):
    # Download text content from the URL
    text = urlopen(url).read().decode("utf-8", errors="ignore")

    # Split text into lines
    lines = text.splitlines()

    # Create empty output lists
    out_year = []
    out_month = []
    out_val = []

    # Read each line
    for line in lines:
        parts = line.strip().split()

        # Expect lines with: year month value
        if len(parts) < 3:
            continue

        # Skip non-data lines
        try:
            year = int(parts[0])
            month = int(parts[1])
            val = float(parts[2])
        except ValueError:
            continue

        out_year.append(year)
        out_month.append(month)
        out_val.append(val)

    # Convert lists to numpy arrays
    return (
        np.array(out_year, dtype=int),
        np.array(out_month, dtype=int),
        np.array(out_val, dtype=float)
    )

def align_to_sst_dates(sst_dates, idx_year, idx_month, idx_val):
    # Convert SST dates like '2021-06' into separate year and month arrays
    sst_year = np.array([int(d[:4]) for d in sst_dates], dtype=int)
    sst_month = np.array([int(d[5:7]) for d in sst_dates], dtype=int)

    # Build YYYYMM integer keys for matching
    sst_key = sst_year * 100 + sst_month
    idx_key = idx_year * 100 + idx_month

    # Fill the aligned output with NaN first
    out = np.full(len(sst_key), np.nan, dtype=float)

    # Match each SST date with the corresponding climate-index value
    for i, key in enumerate(sst_key):
        hit = np.where(idx_key == key)[0]
        if len(hit) > 0:
            out[i] = idx_val[hit[0]]

    return out

def corr_valid(x, y):
    # Keep only entries where both series are finite
    valid = np.isfinite(x) & np.isfinite(y)

    # If too few points remain, return NaN
    if np.sum(valid) < 3:
        return np.nan

    # Compute the Pearson correlation coefficient
    return np.corrcoef(x[valid], y[valid])[0, 1]

def standardize(x):
    # Convert to numpy array
    x = np.array(x, dtype=float)

    # Compute mean and standard deviation while ignoring NaNs
    mu = np.nanmean(x)
    sd = np.nanstd(x)

    # Avoid division by zero
    if not np.isfinite(sd) or sd == 0:
        return np.full_like(x, np.nan, dtype=float)

    # Return standardized values
    return (x - mu) / sd

# Define the remote climate-index URLs
enso_url = "https://psl.noaa.gov/data/correlation/nina34.anom.data"
pdo_url  = "https://psl.noaa.gov/pdo/data/pdo.timeseries.sstens.data"
nao_url  = "https://ftp.cpc.ncep.noaa.gov/wd52dg/data/indices/nao_index.tim"

# Read the remote climate index files
enso_year, enso_month, enso_val = read_psl_table_from_url(enso_url)
pdo_year,  pdo_month,  pdo_val  = read_psl_table_from_url(pdo_url)
nao_year,  nao_month,  nao_val  = read_cpc_index_from_url(nao_url)

# Align the climate indices with the Lake Michigan SST dates
enso_match = align_to_sst_dates(dates_array, enso_year, enso_month, enso_val)
pdo_match  = align_to_sst_dates(dates_array, pdo_year,  pdo_month,  pdo_val)
nao_match  = align_to_sst_dates(dates_array, nao_year,  nao_month,  nao_val)

# Print simple zero-lag correlations
print("ENSO correlation =", corr_valid(sst_use, enso_match))
print("PDO correlation  =", corr_valid(sst_use, pdo_match))
print("NAO correlation  =", corr_valid(sst_use, nao_match))


In [ ]:

# Plot standardized time series so they can be compared on the same scale
x = np.arange(len(dates_array))

plt.figure(figsize=(11, 5))
plt.plot(x, standardize(sst_use), label='Lake Michigan SST anomaly')
plt.plot(x, standardize(enso_match), label='ENSO Niño 3.4')
plt.plot(x, standardize(pdo_match), label='PDO')
plt.plot(x, standardize(nao_match), label='NAO')

tick_step = max(1, len(dates_array) // 8)
plt.xticks(x[::tick_step], dates_array[::tick_step], rotation=45)
plt.ylabel('Standardized value')
plt.title('Lake Michigan SST anomaly vs climate indices')
plt.legend()
plt.tight_layout()
plt.show()



## Regrid Lake Michigan SST to a regular grid

This section creates a target grid, interpolates the native satellite data to that grid,
and applies the lake mask again on the target grid.


In [ ]:

def remap_satellite(lon, lat, data, remap_lon, remap_lat):
    # Flatten the 2D source longitude, latitude, and data arrays into 1D arrays
    x = lon.flatten()
    y = lat.flatten()
    z = data.flatten()

    # Keep only valid values for interpolation
    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x_valid = x[mask]
    y_valid = y[mask]
    z_valid = z[mask]

    # If there are no valid source points, return a target field of NaNs
    if len(z_valid) == 0:
        return np.full(remap_lon.shape, np.nan)

    # Interpolate from the irregular source grid to the regular target grid
    # nearest is used here because it worked well in the original workflow
    new_data = griddata(
        (x_valid, y_valid),
        z_valid,
        (remap_lon, remap_lat),
        method='nearest'
    )

    return new_data


In [ ]:

# Re-open one monthly file as an example for regridding
dataset = nc.Dataset(path + 'GL_ACSPO_AVHRR_merged_monthly_SST_202112.nc', 'r')

# Read the source longitude, latitude, and SST
lon = dataset.variables['longitude'][:]
lat = dataset.variables['latitude'][:]
sst = dataset.variables['sea_surface_temperature'][:] + 27315.0

# Close the file
dataset.close()

# Build the Lake Michigan mask on the native satellite grid
source_mask = make_lake_mask(lon, lat, scale="50m")

# Keep only Lake Michigan values
sst_michigan = np.where(source_mask, sst, np.nan)

# Define a regular target grid over the Lake Michigan region
remap_lon_1d = np.arange(-88.2, -84.8 + 0.02, 0.02)
remap_lat_1d = np.arange(41.5, 46.2 + 0.02, 0.02)

# Turn the 1D target coordinates into 2D meshgrid arrays
remap_lon_2d, remap_lat_2d = np.meshgrid(remap_lon_1d, remap_lat_1d)

# Interpolate the native-grid SST to the regular target grid
sst_michigan_regridded = remap_satellite(
    lon, lat, sst_michigan,
    remap_lon_2d, remap_lat_2d
)

# Build the Lake Michigan mask on the target grid
target_mask = make_lake_mask(remap_lon_2d, remap_lat_2d, scale="50m")

# Apply the target-grid mask so land points become NaN again
sst_michigan_regridded = np.where(target_mask, sst_michigan_regridded, np.nan)

# Print output shape
print('Regridded field shape =', sst_michigan_regridded.shape)


In [ ]:

# Plot the source masked field and the regridded field side by side
fig = plt.figure(figsize=(11, 4))

# Left: native satellite grid
ax1 = fig.add_subplot(1, 2, 1, projection=ccrs.PlateCarree())
pcm1 = ax1.pcolormesh(lon, lat, sst_michigan, shading='auto', transform=ccrs.PlateCarree())
ax1.coastlines(resolution='10m', linewidth=0.5)
ax1.set_extent([-89, -84, 41, 47], crs=ccrs.PlateCarree())
ax1.set_title('Native masked SST')
plt.colorbar(pcm1, ax=ax1, shrink=0.8)

# Right: regular target grid
ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
pcm2 = ax2.pcolormesh(remap_lon_2d, remap_lat_2d, sst_michigan_regridded,
                      shading='auto', transform=ccrs.PlateCarree())
ax2.coastlines(resolution='10m', linewidth=0.5)
ax2.set_extent([-89, -84, 41, 47], crs=ccrs.PlateCarree())
ax2.set_title('Regridded SST')
plt.colorbar(pcm2, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()



## Save the regridded monthly data into a NetCDF file

The workflow below:
- defines a fixed target grid once
- defines the target Lake Michigan mask once
- creates the output NetCDF file first
- loops over all months
- writes each regridded SST field directly into the file

This is more memory-efficient than storing every month in a Python list first.


In [ ]:

# Define the input path and output file
inpath = '/proj/argon/yguo/Data/GL_ACSPO_AVHRR_Daily_Merge_SST/merged_monthly/'
outfile = '/proj/argon/yguo/Data/GL_ACSPO_AVHRR_Daily_Merge_SST/lake_michigan_regridded_monthly_2021_2025.nc'

# Define the regular target grid once
remap_lon_1d = np.arange(-88.2, -84.8 + 0.02, 0.02)
remap_lat_1d = np.arange(41.5, 46.2 + 0.02, 0.02)
remap_lon_2d, remap_lat_2d = np.meshgrid(remap_lon_1d, remap_lat_1d)

# Create the target Lake Michigan mask once
target_mask = make_lake_mask(remap_lon_2d, remap_lat_2d, scale="50m")

# Compute the total number of months from 2021-06 through 2025-08
ntime = (2025 - 2021) * 12 + (8 - 6) + 1

# Get target-grid sizes
nlat = len(remap_lat_1d)
nlon = len(remap_lon_1d)

# Create the output NetCDF file
ds_out = nc.Dataset(outfile, 'w', format='NETCDF4')

# Create dimensions
ds_out.createDimension('time', ntime)
ds_out.createDimension('lat', nlat)
ds_out.createDimension('lon', nlon)

# Create variables
time_var = ds_out.createVariable('time', str, ('time',))
lat_var = ds_out.createVariable('lat', 'f4', ('lat',))
lon_var = ds_out.createVariable('lon', 'f4', ('lon',))
lon2d_var = ds_out.createVariable('longitude', 'f4', ('lat', 'lon'))
lat2d_var = ds_out.createVariable('latitude', 'f4', ('lat', 'lon'))
mask_var = ds_out.createVariable('lake_mask', 'i1', ('lat', 'lon'))
sst_var = ds_out.createVariable(
    'sst_lake_michigan',
    'f4',
    ('time', 'lat', 'lon'),
    zlib=True,
    complevel=4,
    fill_value=np.nan
)

# Write the fixed coordinate variables
lat_var[:] = remap_lat_1d
lon_var[:] = remap_lon_1d
lon2d_var[:, :] = remap_lon_2d
lat2d_var[:, :] = remap_lat_2d
mask_var[:, :] = target_mask.astype(np.int8)

# Add metadata
lat_var.units = 'degrees_north'
lon_var.units = 'degrees_east'
lon2d_var.units = 'degrees_east'
lat2d_var.units = 'degrees_north'
mask_var.long_name = 'Lake Michigan mask'
sst_var.long_name = 'Regridded Lake Michigan SST'
time_var.long_name = 'time as YYYY-MM string'
ds_out.description = 'Monthly regridded Lake Michigan SST from GL_ACSPO_AVHRR merged monthly data'
ds_out.source = 'Generated from native satellite monthly SST files'

# Loop over all available months
for yr in range(2021, 2026):
    for mon in range(1, 13):

        # Skip months before June 2021
        if yr == 2021 and mon < 6:
            continue

        # Skip months after August 2025
        if yr == 2025 and mon > 8:
            continue

        # Compute the time index using June 2021 as the origin
        it = (yr - 2021) * 12 + (mon - 6)

        # Store the string time stamp
        time_var[it] = f'{yr}-{mon:02d}'

        # Build the input filename
        filename = f'GL_ACSPO_AVHRR_merged_monthly_SST_{yr}{mon:02d}.nc'
        filepath = os.path.join(inpath, filename)

        # Skip the month if the file is missing
        if not os.path.exists(filepath):
            print('Missing file:', filepath)
            continue

        print(f'Processing {yr}-{mon:02d} -> time index {it}')

        # Open the monthly file
        dataset = nc.Dataset(filepath, 'r')

        # Read source longitude, latitude, and SST
        lon = dataset.variables['longitude'][:]
        lat = dataset.variables['latitude'][:]
        sst = dataset.variables['sea_surface_temperature'][:] + 27315.0

        # Close the monthly file
        dataset.close()

        # Build the Lake Michigan mask on the source grid
        source_mask = make_lake_mask(lon, lat, scale="50m")

        # Keep only Lake Michigan values
        sst_lake = np.where(source_mask, sst, np.nan)

        # Regrid to the regular target grid
        sst_regrid = remap_satellite(
            lon, lat, sst_lake,
            remap_lon_2d, remap_lat_2d
        )

        # Apply the target Lake Michigan mask
        sst_regrid = np.where(target_mask, sst_regrid, np.nan)

        # Write directly into the output file
        sst_var[it, :, :] = sst_regrid

        # Force a disk write so partial progress is preserved
        ds_out.sync()

# Close the output file after all months are done
ds_out.close()

# Confirm the output path
print('Saved to:', outfile)
